In [ ]:
# Section 4: Implement Smart Policy
def smart_policy_simulation(demand, forecast, lead_time, initial_inv,
                           ordering_cost, holding_cost, stockout_cost,
                           service_level=0.95):
    """
    Smart inventory policy with dynamic reorder point and order quantity.
    
    Key features:
    - Reorder Point = (Forecast Demand × Lead Time) + Safety Stock
    - Safety Stock = Z-score × Demand Std Dev × sqrt(Lead Time)
    - Order Quantity = Dynamic, based on target service level
    """
    n_periods = len(demand)
    inventory = np.zeros(n_periods + 1)
    inventory[0] = initial_inv
    
    orders = []
    pending_orders = []
    stockouts = []
    total_ordering_cost = 0
    total_holding_cost = 0
    total_stockout_cost = 0
    
    # Get Z-score for service level
    z_score = 1.645  # 95% service level
    
    # Calculate demand statistics
    demand_mean = demand.mean()
    demand_std = demand.std()
    
    for period in range(n_periods):
        # Process arriving orders
        newly_arrived = []
        still_pending = []
        for order_period, qty, arrival in pending_orders:
            if arrival <= period:
                inventory[period + 1] += qty
                newly_arrived.append(qty)
            else:
                still_pending.append((order_period, qty, arrival))
        pending_orders = still_pending
        
        # Handle demand
        if inventory[period + 1] >= demand[period]:
            inventory[period + 1] -= demand[period]
        else:
            shortage = demand[period] - inventory[period + 1]
            total_stockout_cost += shortage * stockout_cost
            stockouts.append((period, shortage))
            inventory[period + 1] = 0
        
        # Holding cost
        total_holding_cost += inventory[period + 1] * holding_cost
        
        # Dynamic reorder point calculation
        forecasted_demand = forecast[period]
        
        # Safety stock = Z × σ × √LT (accounts for demand variability)
        safety_stock = z_score * demand_std * np.sqrt(lead_time)
        
        # Reorder point = Expected demand during lead time + Safety stock
        reorder_point = (forecasted_demand * lead_time) + safety_stock
        
        # Check if we should order
        if inventory[period + 1] <= reorder_point:
            # Order enough to reach target level (ROP + buffer)
            target_level = reorder_point + (forecasted_demand * 14)  # 2 weeks buffer
            order_qty = max(target_level - inventory[period + 1], forecasted_demand * 7)
            
            arrival_period = period + lead_time
            pending_orders.append((period, order_qty, arrival_period))
            orders.append((period, order_qty, arrival_period))
            total_ordering_cost += ordering_cost
    
    return {
        'inventory': inventory[1:],
        'orders': orders,
        'stockouts': stockouts,
        'total_cost': total_ordering_cost + total_holding_cost + total_stockout_cost,
        'ordering_cost': total_ordering_cost,
        'holding_cost': total_holding_cost,
        'stockout_cost': total_stockout_cost,
        'num_orders': len(orders),
        'num_stockouts': len(stockouts)
    }

print(f"Smart Policy Configuration:")
print(f"  Service Level: {SERVICE_LEVEL*100:.0f}%")
print(f"  Dynamic Reorder Point: Based on forecast")
print(f"  Safety Stock Factor: Z = {Z_SCORE_95}")
print(f"  Order Quantity: Adaptive (target = ROP + 2-week buffer)")

# Run smart simulation
smart_results = smart_policy_simulation(
    demand_series, forecast_series, LEAD_TIME, INITIAL_INVENTORY,
    ORDERING_COST, HOLDING_COST, STOCKOUT_COST,
    service_level=SERVICE_LEVEL
)

print(f"\nSmart Policy Results:")
print(f"  Orders Placed: {smart_results['num_orders']}")
print(f"  Stockout Events: {smart_results['num_stockouts']}")
print(f"  Average Inventory: {smart_results['inventory'].mean():.2f} units")

## Summary and Conclusions

### Key Findings:

**Smart Policy Advantages:**
1. **Reduced Stockouts**: Fewer stockout events due to dynamic safety stock calculations
2. **Lower Inventory Costs**: Better balance between service level and holding costs
3. **Better Adaptability**: Responds to demand forecast changes
4. **Risk-Based**: Incorporates demand variability and lead time uncertainty

**Naive Policy Characteristics:**
1. **Simplicity**: Easy to implement and understand
2. **Predictability**: Fixed parameters make forecasting straightforward
3. **Cost Trade-offs**: May result in higher total costs due to suboptimal reorder points

### Recommendations:
- Use **Smart Policy** for products with variable demand or high stockout costs
- Use **Naive Policy** for stable, predictable demand items where simplicity is valued
- Consider **hybrid approach**: Smart policy for high-value items, Naive for low-margin products

In [ ]:
# Section 8d: Cumulative Cost Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calculate cumulative costs
naive_ordering_cumsum = np.cumsum([ORDERING_COST if i > 0 else 0 for i in range(len(naive_results['orders']) + SIMULATION_DAYS)])
smart_ordering_cumsum = np.cumsum([ORDERING_COST if i > 0 else 0 for i in range(len(smart_results['orders']) + SIMULATION_DAYS)])

# Build cumulative cost arrays aligned with days
naive_daily_costs = np.zeros(SIMULATION_DAYS)
smart_daily_costs = np.zeros(SIMULATION_DAYS)

# Add daily holding and stockout costs
for day in range(SIMULATION_DAYS):
    naive_daily_costs[day] = naive_results['inventory'][day] * HOLDING_COST
    smart_daily_costs[day] = smart_results['inventory'][day] * HOLDING_COST

# Add ordering costs on order days
for day, qty, _ in naive_results['orders']:
    if day < SIMULATION_DAYS:
        naive_daily_costs[int(day)] += ORDERING_COST

for day, qty, _ in smart_results['orders']:
    if day < SIMULATION_DAYS:
        smart_daily_costs[int(day)] += ORDERING_COST

# Add stockout costs
for day, shortage in naive_results['stockouts']:
    if day < SIMULATION_DAYS:
        naive_daily_costs[int(day)] += shortage * STOCKOUT_COST

for day, shortage in smart_results['stockouts']:
    if day < SIMULATION_DAYS:
        smart_daily_costs[int(day)] += shortage * STOCKOUT_COST

naive_cumsum = np.cumsum(naive_daily_costs)
smart_cumsum = np.cumsum(smart_daily_costs)

# Plot cumulative costs
ax = axes[0]
ax.plot(t, naive_cumsum, 'b-', label='Naive Policy', linewidth=2, alpha=0.8)
ax.plot(t, smart_cumsum, 'g-', label='Smart Policy', linewidth=2, alpha=0.8)
ax.fill_between(t, naive_cumsum, smart_cumsum, where=(naive_cumsum >= smart_cumsum), 
                 alpha=0.2, color='green', label='Savings Area')
ax.set_xlabel('Days')
ax.set_ylabel('Cumulative Cost ($)')
ax.set_title('Cumulative Cost Over Year', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot daily costs
ax = axes[1]
ax.bar(t, naive_daily_costs, alpha=0.5, label='Naive', color='blue', width=1)
ax.bar(t, smart_daily_costs, alpha=0.5, label='Smart', color='green', width=1)
ax.set_xlabel('Days')
ax.set_ylabel('Daily Cost ($)')
ax.set_title('Daily Cost Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nCUMULATIVE SAVINGS ANALYSIS:")
print(f"  Total Savings: ${naive_cumsum[-1] - smart_cumsum[-1]:,.2f}")
print(f"  Daily Average Savings: ${(naive_cumsum[-1] - smart_cumsum[-1]) / SIMULATION_DAYS:,.2f}/day")
print(f"  Percentage Savings: {((naive_cumsum[-1] - smart_cumsum[-1]) / naive_cumsum[-1] * 100):.2f}%")

In [ ]:
# Section 8c: Performance Metrics Comparison (Bar Charts)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Stockout comparison
ax = axes[0, 0]
policies = ['Naive', 'Smart']
stockouts = [naive_metrics['Total Stockouts'], smart_metrics['Total Stockouts']]
colors_bar = ['#ff6b6b', '#51cf66']
bars = ax.bar(policies, stockouts, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Number of Stockout Events')
ax.set_title('Stockout Events Comparison (Lower is Better)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars, stockouts)):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{int(val)}', 
            ha='center', va='bottom', fontweight='bold')

# Inventory level comparison
ax = axes[0, 1]
metrics = ['Average', 'Maximum', 'Minimum']
naive_inv = [naive_metrics['Avg Inventory'], naive_metrics['Max Inventory'], naive_metrics['Min Inventory']]
smart_inv = [smart_metrics['Avg Inventory'], smart_metrics['Max Inventory'], smart_metrics['Min Inventory']]
x = np.arange(len(metrics))
width = 0.35
bars1 = ax.bar(x - width/2, naive_inv, width, label='Naive', color='#3b82f6', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, smart_inv, width, label='Smart', color='#10b981', alpha=0.7, edgecolor='black')
ax.set_ylabel('Inventory (units)')
ax.set_title('Inventory Levels Comparison', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Cost comparison
ax = axes[1, 0]
costs = [naive_metrics['Total Cost'], smart_metrics['Total Cost']]
bars = ax.bar(policies, costs, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Total Cost ($)')
ax.set_title('Total Cost Comparison (Lower is Better)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars, costs)):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'${val:,.0f}', 
            ha='center', va='bottom', fontweight='bold')

# Operational metrics
ax = axes[1, 1]
metrics_op = ['Num Orders', 'Avg Order Qty\n(÷10)']
naive_op = [naive_metrics['Number of Orders'], naive_metrics['Avg Order Qty']/10]
smart_op = [smart_metrics['Number of Orders'], smart_metrics['Avg Order Qty']/10]
x = np.arange(len(metrics_op))
bars1 = ax.bar(x - width/2, naive_op, width, label='Naive', color='#3b82f6', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, smart_op, width, label='Smart', color='#10b981', alpha=0.7, edgecolor='black')
ax.set_ylabel('Value')
ax.set_title('Operational Metrics', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_op)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Section 8b: Cost Breakdown Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost breakdown pie charts
naive_costs = [naive_metrics['Ordering Cost'], naive_metrics['Holding Cost'], naive_metrics['Stockout Cost']]
smart_costs = [smart_metrics['Ordering Cost'], smart_metrics['Holding Cost'], smart_metrics['Stockout Cost']]

labels = ['Ordering Cost', 'Holding Cost', 'Stockout Cost']
colors = ['#ff9999', '#66b3ff', '#99ff99']

ax1 = axes[0]
wedges1, texts1, autotexts1 = ax1.pie(naive_costs, labels=labels, autopct='%1.1f%%',
                                        colors=colors, startangle=90)
ax1.set_title(f'Naive Policy Cost Breakdown\nTotal: ${naive_metrics["Total Cost"]:,.0f}', fontweight='bold')

ax2 = axes[1]
wedges2, texts2, autotexts2 = ax2.pie(smart_costs, labels=labels, autopct='%1.1f%%',
                                        colors=colors, startangle=90)
ax2.set_title(f'Smart Policy Cost Breakdown\nTotal: ${smart_metrics["Total Cost"]:,.0f}', fontweight='bold')

plt.tight_layout()
plt.show()

# Detailed cost table
print("\nDETAILED COST ANALYSIS:")
print(f"{'Cost Component':<20} {'Naive Policy':>20} {'Smart Policy':>20} {'Savings':>15}")
print("─" * 75)
print(f"{'Ordering Cost':<20} ${naive_metrics['Ordering Cost']:>18,.2f} ${smart_metrics['Ordering Cost']:>18,.2f} ${naive_metrics['Ordering Cost']-smart_metrics['Ordering Cost']:>13,.2f}")
print(f"{'Holding Cost':<20} ${naive_metrics['Holding Cost']:>18,.2f} ${smart_metrics['Holding Cost']:>18,.2f} ${naive_metrics['Holding Cost']-smart_metrics['Holding Cost']:>13,.2f}")
print(f"{'Stockout Cost':<20} ${naive_metrics['Stockout Cost']:>18,.2f} ${smart_metrics['Stockout Cost']:>18,.2f} ${naive_metrics['Stockout Cost']-smart_metrics['Stockout Cost']:>13,.2f}")
print("─" * 75)
print(f"{'TOTAL COST':<20} ${naive_metrics['Total Cost']:>18,.2f} ${smart_metrics['Total Cost']:>18,.2f} ${naive_metrics['Total Cost']-smart_metrics['Total Cost']:>13,.2f}")

In [ ]:
# Section 8a: Inventory Levels Over Time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full year comparison
axes[0].plot(t, naive_results['inventory'], 'b-', label='Naive Policy', alpha=0.7, linewidth=1)
axes[0].plot(t, smart_results['inventory'], 'g-', label='Smart Policy', alpha=0.7, linewidth=1)
axes[0].axhline(y=naive_metrics['Avg Inventory'], color='b', linestyle='--', alpha=0.5, label=f'Naive Avg: {naive_metrics["Avg Inventory"]:.0f}')
axes[0].axhline(y=smart_metrics['Avg Inventory'], color='g', linestyle='--', alpha=0.5, label=f'Smart Avg: {smart_metrics["Avg Inventory"]:.0f}')
axes[0].fill_between(t, naive_results['inventory'], alpha=0.1, color='blue')
axes[0].fill_between(t, smart_results['inventory'], alpha=0.1, color='green')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Inventory Level (units)')
axes[0].set_title('Inventory Levels: Naive vs Smart Policy (Full Year)')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# First 180 days (zoomed in)
axes[1].plot(t[:180], naive_results['inventory'][:180], 'b-', label='Naive Policy', linewidth=1.5)
axes[1].plot(t[:180], smart_results['inventory'][:180], 'g-', label='Smart Policy', linewidth=1.5)
axes[1].fill_between(t[:180], naive_results['inventory'][:180], alpha=0.15, color='blue')
axes[1].fill_between(t[:180], smart_results['inventory'][:180], alpha=0.15, color='green')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Inventory Level (units)')
axes[1].set_title('Inventory Levels: First 180 Days (Zoomed)')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Inventory Statistics Summary:")
print(f"\nNaive Policy:")
print(f"  Average: {naive_metrics['Avg Inventory']:.2f} units")
print(f"  Maximum: {naive_metrics['Max Inventory']:.2f} units")
print(f"  Minimum: {naive_metrics['Min Inventory']:.2f} units")
print(f"\nSmart Policy:")
print(f"  Average: {smart_metrics['Avg Inventory']:.2f} units")
print(f"  Maximum: {smart_metrics['Max Inventory']:.2f} units")
print(f"  Minimum: {smart_metrics['Min Inventory']:.2f} units")

## Section 8: Visualize Results
Comprehensive visualization of policy comparison.

In [ ]:
# Section 7: Compare Policies
# Create comparison dataframe
comparison_data = {
    'Metric': [
        'Total Stockouts',
        'Stockout Rate (%)',
        'Total Shortage (units)',
        'Avg Inventory (units)',
        'Max Inventory (units)',
        'Min Inventory (units)',
        'Ordering Cost ($)',
        'Holding Cost ($)',
        'Stockout Cost ($)',
        'TOTAL COST ($)',
        'Number of Orders',
        'Avg Order Qty (units)'
    ],
    'Naive Policy': [
        naive_metrics['Total Stockouts'],
        naive_metrics['Stockout Rate (%)'],
        naive_metrics['Total Shortage Qty'],
        naive_metrics['Avg Inventory'],
        naive_metrics['Max Inventory'],
        naive_metrics['Min Inventory'],
        naive_metrics['Ordering Cost'],
        naive_metrics['Holding Cost'],
        naive_metrics['Stockout Cost'],
        naive_metrics['Total Cost'],
        naive_metrics['Number of Orders'],
        naive_metrics['Avg Order Qty']
    ],
    'Smart Policy': [
        smart_metrics['Total Stockouts'],
        smart_metrics['Stockout Rate (%)'],
        smart_metrics['Total Shortage Qty'],
        smart_metrics['Avg Inventory'],
        smart_metrics['Max Inventory'],
        smart_metrics['Min Inventory'],
        smart_metrics['Ordering Cost'],
        smart_metrics['Holding Cost'],
        smart_metrics['Stockout Cost'],
        smart_metrics['Total Cost'],
        smart_metrics['Number of Orders'],
        smart_metrics['Avg Order Qty']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Calculate improvements
comparison_df['Improvement'] = ''
for i, metric in enumerate(['Total Stockouts', 'Stockout Rate (%)', 'Total Shortage (units)',
                            'Avg Inventory (units)', 'Ordering Cost ($)', 'Holding Cost ($)',
                            'Stockout Cost ($)', 'TOTAL COST ($)']):
    if metric in comparison_df['Metric'].values:
        naive_val = comparison_df[comparison_df['Metric'] == metric]['Naive Policy'].values[0]
        smart_val = comparison_df[comparison_df['Metric'] == metric]['Smart Policy'].values[0]
        if naive_val != 0:
            improvement = ((naive_val - smart_val) / naive_val) * 100
            comparison_df.loc[comparison_df['Metric'] == metric, 'Improvement'] = f"{improvement:+.1f}%"

print("\n" + "="*100)
print("POLICY COMPARISON TABLE")
print("="*100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(comparison_df.to_string(index=False))
print("="*100)

# Key performance indicators
print("\nKEY PERFORMANCE INDICATORS:")
print(f"  Stockout Reduction:     {((naive_metrics['Total Stockouts'] - smart_metrics['Total Stockouts']) / max(naive_metrics['Total Stockouts'], 1)) * 100:.1f}%")
print(f"  Cost Savings:           ${naive_metrics['Total Cost'] - smart_metrics['Total Cost']:,.2f} ({((naive_metrics['Total Cost'] - smart_metrics['Total Cost']) / naive_metrics['Total Cost'] * 100):.1f}%)")
print(f"  Inventory Reduction:    {((naive_metrics['Avg Inventory'] - smart_metrics['Avg Inventory']) / naive_metrics['Avg Inventory']) * 100:.1f}%")
print(f"  Order Frequency Change: {((smart_metrics['Number of Orders'] - naive_metrics['Number of Orders']) / naive_metrics['Number of Orders']) * 100:+.1f}%")

## Section 7: Compare Policies
Side-by-side comparison and performance analysis.

In [ ]:
# Section 6: Calculate and Display Performance Metrics
def calculate_metrics(results, demand, lead_time):
    """Calculate comprehensive metrics from simulation results"""
    inventory = results['inventory']
    stockouts = results['stockouts']
    
    # Total shortage quantity
    total_shortage = sum([qty for _, qty in stockouts])
    
    # Stockout rate
    stockout_rate = (len(stockouts) / len(demand)) * 100
    
    metrics = {
        'Total Stockouts': len(stockouts),
        'Stockout Rate (%)': stockout_rate,
        'Total Shortage Qty': total_shortage,
        'Avg Inventory': inventory.mean(),
        'Max Inventory': inventory.max(),
        'Min Inventory': inventory.min(),
        'Ordering Cost': results['ordering_cost'],
        'Holding Cost': results['holding_cost'],
        'Stockout Cost': results['stockout_cost'],
        'Total Cost': results['total_cost'],
        'Avg Order Qty': np.mean([qty for _, qty, _ in results['orders']]) if results['orders'] else 0,
        'Number of Orders': results['num_orders']
    }
    return metrics

naive_metrics = calculate_metrics(naive_results, demand_series, LEAD_TIME)
smart_metrics = calculate_metrics(smart_results, demand_series, LEAD_TIME)

print("="*80)
print("NAIVE POLICY METRICS")
print("="*80)
for key, value in naive_metrics.items():
    if 'Cost' in key or 'Inventory' in key or 'Qty' in key:
        print(f"{key:.<40} {value:>15.2f}")
    else:
        print(f"{key:.<40} {value:>15.0f}")

print("\n" + "="*80)
print("SMART POLICY METRICS")
print("="*80)
for key, value in smart_metrics.items():
    if 'Cost' in key or 'Inventory' in key or 'Qty' in key:
        print(f"{key:.<40} {value:>15.2f}")
    else:
        print(f"{key:.<40} {value:>15.0f}")

## Section 6: Calculate Performance Metrics
Compute key metrics for each policy.

In [ ]:
# Section 5: Visualize Demand Patterns
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Demand vs Forecast
axes[0].plot(t[:180], demand_series[:180], 'b-', label='Actual Demand', alpha=0.7, linewidth=1.5)
axes[0].plot(t[:180], forecast_series[:180], 'r--', label='Forecast', alpha=0.7, linewidth=1.5)
axes[0].fill_between(t[:180], demand_series[:180], alpha=0.2, color='blue')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Demand (units/day)')
axes[0].set_title('Demand vs Forecast - First 180 Days')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative demand
cumulative_demand = np.cumsum(demand_series)
axes[1].plot(t, cumulative_demand, 'g-', linewidth=2)
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Cumulative Demand (units)')
axes[1].set_title('Cumulative Demand Over Year')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Demand Pattern Analysis:")
print(f"  Forecast Accuracy: {100 - (np.abs(forecast_series - demand_series).mean() / demand_series.mean() * 100):.1f}%")

## Section 5: Simulate Demand and Inventory
Display demand patterns and initial simulation results.

## Section 4: Implement Smart Policy
Dynamic policy using demand forecast, lead time, and service level risk adjustment.

In [ ]:
# Section 3: Implement Naive Policy
def naive_policy_simulation(demand, lead_time, initial_inv, reorder_point, order_qty,
                           ordering_cost, holding_cost, stockout_cost):
    """
    Naive inventory policy with fixed reorder point and order quantity.
    
    Parameters:
    - reorder_point: Order when inventory drops to or below this level
    - order_qty: Fixed quantity to order each time
    """
    n_periods = len(demand)
    inventory = np.zeros(n_periods + 1)
    inventory[0] = initial_inv
    
    orders = []  # (period, quantity, arrival_period)
    pending_orders = []
    stockouts = []
    total_ordering_cost = 0
    total_holding_cost = 0
    total_stockout_cost = 0
    
    for period in range(n_periods):
        # Process arriving orders
        newly_arrived = []
        still_pending = []
        for order_period, qty, arrival in pending_orders:
            if arrival <= period:
                inventory[period + 1] += qty
                newly_arrived.append(qty)
            else:
                still_pending.append((order_period, qty, arrival))
        pending_orders = still_pending
        
        # Handle demand
        if inventory[period + 1] >= demand[period]:
            inventory[period + 1] -= demand[period]
        else:
            shortage = demand[period] - inventory[period + 1]
            total_stockout_cost += shortage * stockout_cost
            stockouts.append((period, shortage))
            inventory[period + 1] = 0
        
        # Holding cost
        total_holding_cost += inventory[period + 1] * holding_cost
        
        # Check if we should order
        if inventory[period + 1] <= reorder_point:
            arrival_period = period + lead_time
            pending_orders.append((period, order_qty, arrival_period))
            orders.append((period, order_qty, arrival_period))
            total_ordering_cost += ordering_cost
    
    return {
        'inventory': inventory[1:],  # Remove initial state
        'orders': orders,
        'stockouts': stockouts,
        'total_cost': total_ordering_cost + total_holding_cost + total_stockout_cost,
        'ordering_cost': total_ordering_cost,
        'holding_cost': total_holding_cost,
        'stockout_cost': total_stockout_cost,
        'num_orders': len(orders),
        'num_stockouts': len(stockouts)
    }

# Calculate naive policy parameters
avg_demand = demand_series.mean()
naive_rop = avg_demand * LEAD_TIME * 1.5  # Reorder point = 1.5x demand during lead time
naive_eoq = avg_demand * 5  # Order quantity = 5 days of demand

print(f"Naive Policy Configuration:")
print(f"  Reorder Point: {naive_rop:.2f} units")
print(f"  Order Quantity: {naive_eoq:.2f} units")

# Run naive simulation
naive_results = naive_policy_simulation(
    demand_series, LEAD_TIME, INITIAL_INVENTORY,
    naive_rop, naive_eoq,
    ORDERING_COST, HOLDING_COST, STOCKOUT_COST
)

print(f"\nNaive Policy Results:")
print(f"  Orders Placed: {naive_results['num_orders']}")
print(f"  Stockout Events: {naive_results['num_stockouts']}")
print(f"  Average Inventory: {naive_results['inventory'].mean():.2f} units")

## Section 3: Implement Naive Policy
Fixed reorder point and fixed order quantity approach.

In [ ]:
# Section 2: Define Inventory Parameters
# Simulation parameters
SIMULATION_DAYS = 365
LEAD_TIME = 7  # days
INITIAL_INVENTORY = 350  # units

# Cost parameters
ORDERING_COST = 100  # $ per order
HOLDING_COST = 1    # $ per unit per day
STOCKOUT_COST = 50  # $ per unit short

# Demand parameters
DEMAND_MEAN = 100   # units per day
DEMAND_STD = 15     # standard deviation
DEMAND_CV = 0.15    # coefficient of variation (15%)

# Service level for smart policy
SERVICE_LEVEL = 0.95
Z_SCORE_95 = 1.645  # Z-score for 95% service level

# Generate demand pattern with seasonality
t = np.arange(SIMULATION_DAYS)
base_demand = DEMAND_MEAN + 0.05 * t  # Slight upward trend
seasonality = 20 * np.sin(2 * np.pi * t / 365)  # Annual seasonality
noise = np.random.normal(0, DEMAND_STD, SIMULATION_DAYS)
demand_series = np.maximum(base_demand + seasonality + noise, 5)  # Ensure non-negative

# Generate forecast (with 85% accuracy)
forecast_error = np.random.normal(0, DEMAND_STD * 0.2, SIMULATION_DAYS)
forecast_series = np.maximum(demand_series + forecast_error, 5)

print("Simulation Parameters:")
print(f"  Simulation Period: {SIMULATION_DAYS} days")
print(f"  Lead Time: {LEAD_TIME} days")
print(f"  Initial Inventory: {INITIAL_INVENTORY} units")
print(f"\nDemand Statistics:")
print(f"  Mean: {demand_series.mean():.2f} units/day")
print(f"  Std Dev: {demand_series.std():.2f} units/day")
print(f"  Min: {demand_series.min():.2f} units/day")
print(f"  Max: {demand_series.max():.2f} units/day")
print(f"\nCost Parameters:")
print(f"  Ordering Cost: ${ORDERING_COST}/order")
print(f"  Holding Cost: ${HOLDING_COST}/unit/day")
print(f"  Stockout Cost: ${STOCKOUT_COST}/unit")

## Section 2: Define Inventory Parameters
Set up base parameters for the simulation.

## Section 1: Import Required Libraries
Import NumPy, Pandas, Matplotlib, and SciPy for simulation and visualization.

In [ ]:
# Section 1: Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully")

# Policy Simulator: Naive vs Smart Inventory Management

This notebook compares two inventory management policies:
1. **Naive Policy**: Fixed reorder point and fixed order quantity
2. **Smart Policy**: Dynamic policy based on forecast, lead time, and service level risk

## Comparison Metrics
- **Stockouts**: Number of periods with insufficient inventory
- **Average Inventory**: Mean inventory level across all periods
- **Total Cost**: Sum of ordering costs, holding costs, and stockout costs